# Add the Rybizki fidelity dataset

This notebook downloads the Rybizki Gaia fidelity catalog, unpacks the per-level `.npy` files, and turns the fidelity values into a Dask DataFrame keyed by `source_id`.

The main practical goal is to prepare a table with two columns:

- `source_id`: the Gaia source identifier, stored as `int64`.
- `fidelity_v2`: the fidelity value, stored as `float32` to keep memory use down.

The later cells assume that a Dask distributed client named `c` already exists in the notebook session. For example, create one with `c = Client()` for a local cluster or connect to a shared scheduler before scattering the partitions.

In [ ]:
import numpy as np

## Download the catalog archive

The source archive is downloaded directly into the working directory as `fidelities.zip`. Run this from a machine or notebook environment with enough local disk space for both the zip file and the extracted `fidelities/` directory.

In [ ]:
! wget "https://keeper.mpdl.mpg.de/d/21d3582c0df94e19921d/files/?p=%2Fwhole_catalog_v2%2Ffidelities.zip&dl=1" -O fidelities.zip


In [ ]:
! python3 -c "import zipfile; zipfile.ZipFile('fidelities.zip').extractall('.')"

## Inspect one file

Before building a distributed table, inspect a representative `.npy` file to confirm the available fields and dtypes. The notebook expects each file to contain `source_id` and `fidelity_v2`.

In [ ]:
# inspect the data
arr = np.load('fidelities/lvl5_12205.npy')
print(arr.dtype.names)
# ('source_id', 'fidelity_v2')

## Build coarse partitions locally

The fidelity catalog is split across many small `.npy` files. Submitting every file as its own Dask task would create a very large task graph and spend too much time in scheduler overhead. This cell groups the files into a smaller number of pandas DataFrames first, then uses those DataFrames as the eventual Dask partitions.

The casts are intentional: `source_id` becomes a stable integer key, while `fidelity_v2` is kept as `float32` to reduce the amount of data scattered to workers.

In [ ]:
import glob
import numpy as np
import pandas as pd
import dask.dataframe as dd
from dask import delayed
import time

# Grab all files
files = sorted(glob.glob('fidelities/lvl5_*.npy'))


def load_one(fpath):
    arr = np.load(fpath)
    return pd.DataFrame({
        'source_id': arr['source_id'].astype('int64'),
        'fidelity_v2': arr['fidelity_v2'].astype('float32'),
    })

# Batch files into ~200 chunks rather than scattering 12,000 tiny pieces —
# reduces scheduler overhead significantly
n_chunks = 200
chunks = np.array_split(files, n_chunks)

_t1 = time.time()

partitions = []
for chunk_files in chunks:
    dfs = [load_one(f) for f in chunk_files]
    partitions.append(pd.concat(dfs, ignore_index=True))

print('Time elapsed: ',time.time()-_t1)



## Scatter partitions to workers

This step sends the prepared pandas partitions to the connected Dask cluster. It uses small batches so the client does not try to push every partition at once. 

In [ ]:
from dask.distributed import Client

# this is specific to AstroFlow's Dask Operator, which is running in the same Kubernetes cluster as this notebook
c = Client('tcp://simple-scheduler.dask-operator.svc.cluster.local:8786')
print(c)

In [ ]:
futures = []
batch_size = 10  # scatter a handful at a time, not all 200 at once
for i in range(0, len(partitions), batch_size):
    batch = partitions[i:i+batch_size]
    futures.extend(c.scatter(batch))

## Wrap futures as a Dask DataFrame

Each scattered pandas DataFrame is represented by a future. Wrapping the futures with `delayed` lets `dd.from_delayed` assemble them into one logical Dask DataFrame. Supplying `meta` gives Dask the schema up front, so it does not need to execute a sample task just to infer column names and dtypes.

In [ ]:
import dask.dataframe as dd
from dask import delayed

# Wrap each future as a delayed object so dask.dataframe knows how to treat it
delayed_dfs = [delayed(f) for f in futures]

# Need meta - the empty-frame schema - to avoid dask inspecting data itself
meta = partitions[0].iloc[:0]  # empty DataFrame with correct columns/dtypes

ddf = dd.from_delayed(delayed_dfs, meta=meta)

print(ddf.npartitions)  # should match len(futures)
print(ddf.dtypes)

## Index by Gaia source id

Setting `source_id` as the index makes later joins and lookups by Gaia source id more natural. This is also the expensive distributed step: Dask has to shuffle rows between partitions so that the index ordering and partition divisions are consistent.

In [ ]:
_t1 = time.time()

ddf = ddf.set_index('source_id')  # shuffle happens here

print('Time elapsed: ',time.time()-_t1)


## Preview the result

`head()` pulls a small sample back to the notebook process. At this point, check that the index is `source_id`, the `fidelity_v2` column is present, and the values look plausible before writing the dataset out or joining it with other Gaia tables.

In [ ]:
ddf.head()

## Cross-match against Gaia RVS

The next cells join the fidelity table to a filtered subset of Gaia DR3 sources with radial velocities. This gives a working sample with usable astrometry, an available `radial_velocity`, and the Rybizki `fidelity_v2` value attached by `source_id`.

The Gaia source data is read from parquet using `DASK_DATA_PATH_GAIA_DR3_SSD`, so that environment variable needs to point at the local or mounted Gaia DR3 data root.

In [ ]:
from pathlib import Path
import os

source_cols = ['source_id',
               'pmra', 'pmdec', 'parallax_over_error',
               'parallax_error','parallax_pseudocolour_corr',
              'astrometric_params_solved',
              'phot_g_mean_mag',
              'ruwe',
              'visibility_periods_used',
              'astrometric_matched_transits',
              'radial_velocity',
              'rv_template_teff',
              'grvs_mag',
              'random_index']


gaia_source = dd.read_parquet(Path(os.environ['DASK_DATA_PATH_GAIA_DR3_SSD']).joinpath('GDR3_GAIA_SOURCE/'), 
                           columns=source_cols)


# split into two steps to reduce peak memory
gaia_source = gaia_source[
    (gaia_source["ruwe"] < 1.5)
    & (gaia_source["parallax_over_error"] > 3.0)].persist()

gaia_source = gaia_source[
    gaia_source["radial_velocity"].notnull()
].persist()

## Attach fidelity values

Only the columns needed for the cross-match and quality checks are loaded from Gaia. The filters are persisted in two stages to keep the working set smaller before the join. The inner merge keeps sources that appear in both the filtered Gaia RVS sample and the fidelity catalog.

In [ ]:
# set up to do the merge
result = gaia_source.merge(ddf, on="source_id", how="inner")

## Materialize the matched sample

The merge is still lazy until `compute()` runs. This brings the matched subset back into the notebook process as a pandas DataFrame, so it is best done after the Gaia filters have made the result small enough to fit in memory.

In [ ]:
_t1 = time.time()

smalldf = result.compute()

print('Time elapsed: ',time.time()-_t1)

# this has been chopped down a bit already, but let's just see how much time it takes
# took 101s to do the merge, but without the partition mapping will probably take longer

## Preview the cross-match

A quick `head()` check should show Gaia columns plus the joined `fidelity_v2` value. This is a lightweight sanity check before summarizing the fidelity distribution.

In [ ]:
smalldf.head()

## Locate the 0.5 fidelity threshold

To understand how strict a `fidelity_v2 = 0.5` threshold is for this matched sample, compute the empirical percentile rank of 0.5 among finite fidelity values. This reports the percentile closest to the threshold rather than guessing a fixed percentile by hand.

In [ ]:
target_fidelity = 0.5
fidelity = smalldf['fidelity_v2'].to_numpy()
finite_fidelity = fidelity[np.isfinite(fidelity)]
if len(finite_fidelity) == 0:
    raise ValueError('No finite fidelity_v2 values found in smalldf')

sorted_fidelity = np.sort(finite_fidelity)
percentile_at_target = 100.0 * np.searchsorted(
    sorted_fidelity,
    target_fidelity,
    side='right',
) / len(sorted_fidelity)
nearest_percentile_value = np.nanpercentile(
    finite_fidelity,
    percentile_at_target,
)

print(f'fidelity_v2 = {target_fidelity} is closest to the {percentile_at_target:.3f}th percentile')
print(f'np.nanpercentile(..., {percentile_at_target:.3f}) = {nearest_percentile_value:.6f}')

Running this notebook results in the discovery that only 4% of the entire RVS dataset is flagged as having bad astrometry!